# 05 — Strategy comparison dashboard

Capstone notebook that runs all three strategies against the full 2015–2024
universe, generates publication-quality comparison figures, and exports them
to `analysis/figures/` as PNG and PDF.

**Figures produced:**
| File | Description |
|------|-------------|
| `01_equity_curves` | Normalized portfolio value (base=100) for all strategies vs SPY |
| `02_drawdowns` | Drawdown depth comparison across strategies |
| `03_metrics_comparison` | Side-by-side 8-metric performance bar charts |
| `04_rolling_metrics` | 63-day rolling return, Sharpe, and max drawdown |
| `05_correlation_matrix` | Pearson return-correlation heatmap |
| `06_monthly_returns_*` | Monthly return calendar heatmap per strategy |
| `07_trade_analysis_*` | Trade cost and size breakdown per strategy |
| `08_positions_*` | Position weight concentration per strategy |
| `09_combined_portfolio` | Equal-weight combined portfolio vs individuals |
| `10_combined_monthly_returns` | Combined portfolio monthly heatmap |

In [ ]:
%matplotlib inline
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on sys.path when running from notebooks/
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from analysis import (
    plot_equity_curves,
    plot_drawdowns,
    plot_metrics_comparison,
    plot_rolling_metrics,
    plot_correlation_matrix,
    plot_monthly_returns_heatmap,
    plot_trade_analysis,
    plot_position_concentration,
    save_figure,
)
from backtester import (
    DataLoader, Backtester, BacktestResult, compute_metrics,
)
from strategies import (
    MomentumStrategy,
    MeanReversionStrategy,
    MLSignalStrategy,
)
from config import (
    DEFAULT_TICKERS, BENCHMARK_TICKER,
    START_DATE, END_DATE, INITIAL_CAPITAL,
)

FIGURES_DIR = pathlib.Path("../analysis/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {FIGURES_DIR.resolve()}")

In [ ]:
all_tickers = DEFAULT_TICKERS + [BENCHMARK_TICKER]
loader = DataLoader(all_tickers, START_DATE, END_DATE)
prices = loader.load_wide()
print(f"Loaded {prices.shape[1]} tickers x {prices.shape[0]} trading days")
print(f"Date range: {prices.index[0].date()} -> {prices.index[-1].date()}")
prices.head()

In [ ]:
bt = Backtester(prices, config={"allow_short": False})

print("Running momentum strategy...")
mom_result = bt.run(MomentumStrategy(
    lookback_months=12, skip_months=1, n_long=5))

print("\nRunning mean reversion strategy...")
mr_result = bt.run(MeanReversionStrategy(
    bb_window=20, bb_std=2.0, rsi_oversold=35))

print("\nRunning ML signal strategy...")
ml_result = bt.run(MLSignalStrategy(
    model_type="xgboost", min_train_years=3,
    retrain_freq_months=3))

print("\nRunning benchmark (SPY buy-and-hold)...")
spy_result = bt.run_benchmark()

all_results = [mom_result, mr_result, ml_result]
print("\nAll strategies complete.")

In [ ]:
metrics_keys = [
    "total_return_pct", "annualized_return_pct",
    "annualized_volatility_pct", "sharpe_ratio",
    "sortino_ratio", "calmar_ratio", "max_drawdown_pct",
    "max_drawdown_duration_days", "recovery_days",
    "alpha_pct", "beta", "information_ratio",
    "n_trades", "total_cost_pct", "turnover_annual_pct",
]

rows = {}
for r in all_results + [spy_result]:
    rows[r.strategy_name] = {k: r.metrics.get(k) for k in metrics_keys}

summary_df = pd.DataFrame(rows).T
summary_df.index.name = "Strategy"

print("=== Raw metrics (copy-paste into README) ===")
print(summary_df.round(2).to_string())

# Higher-is-better metrics
higher_better = [
    "total_return_pct", "annualized_return_pct",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio",
    "alpha_pct", "information_ratio",
]
# Lower-is-better metrics
lower_better = [
    "max_drawdown_pct", "annualized_volatility_pct", "total_cost_pct",
]

styled = (
    summary_df.round(2)
    .style
    .highlight_max(
        subset=[c for c in higher_better if c in summary_df.columns],
        color="lightgreen", axis=0,
    )
    .highlight_min(
        subset=[c for c in lower_better if c in summary_df.columns],
        color="lightgreen", axis=0,
    )
    .format("{:.2f}")
)
styled

## Figure 1 — Equity curves

In [ ]:
fig, ax = plot_equity_curves(
    all_results, spy_result,
    title="Momentum vs Mean Reversion vs ML Signal vs SPY")
save_figure(fig, "01_equity_curves", FIGURES_DIR)
plt.show()

## Figure 2 — Drawdown comparison

In [ ]:
fig, ax = plot_drawdowns(all_results, spy_result)
save_figure(fig, "02_drawdowns", FIGURES_DIR)
plt.show()

## Figure 3 — Metrics comparison

In [ ]:
fig, axes = plot_metrics_comparison(all_results, spy_result)
save_figure(fig, "03_metrics_comparison", FIGURES_DIR)
plt.show()

## Figure 4 — Rolling performance

In [ ]:
fig, axes = plot_rolling_metrics(all_results, spy_result, window=63)
save_figure(fig, "04_rolling_metrics", FIGURES_DIR)
plt.show()

## Figure 5 — Return correlation

In [ ]:
fig, ax = plot_correlation_matrix(all_results, spy_result)
save_figure(fig, "05_correlation_matrix", FIGURES_DIR)
plt.show()

# Print raw correlation matrix
corr_labels = {r.strategy_name[:20]: r.returns for r in all_results + [spy_result]}
corr_df = pd.concat(corr_labels, axis=1).dropna().corr()
print("\nCorrelation matrix:")
print(corr_df.round(4).to_string())

# Find least-correlated pair
n = len(corr_df)
off_diag = [
    (corr_df.columns[i], corr_df.columns[j], corr_df.iloc[i, j])
    for i in range(n) for j in range(n) if i != j
]
min_pair = min(off_diag, key=lambda x: x[2])
print(f"\nLeast correlated pair: {min_pair[0]} <-> {min_pair[1]} (r={min_pair[2]:.3f})")
print("-> A combined portfolio of these two would benefit from meaningful diversification.")

## Figure 6 — Monthly returns heatmaps

In [ ]:
for result in all_results:
    fig, ax = plot_monthly_returns_heatmap(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"06_monthly_returns_{name}", FIGURES_DIR)
    plt.show()

## Figure 7 — Trade analysis

In [ ]:
for result in all_results:
    fig, axes = plot_trade_analysis(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"07_trade_analysis_{name}", FIGURES_DIR)
    plt.show()

## Figure 8 — Position concentration

In [ ]:
for result in all_results:
    fig, axes = plot_position_concentration(result)
    name = result.strategy_name.split("(")[0].strip().lower().replace(" ", "_")
    save_figure(fig, f"08_positions_{name}", FIGURES_DIR)
    plt.show()

## Portfolio-level analysis

Simulate a naive equal-weight combination of all three strategies to
quantify the diversification benefit.

In [ ]:
# Equal-weight average of all three strategy return streams
combined_returns = pd.concat(
    [r.returns for r in all_results], axis=1
).mean(axis=1)  # equal weight across strategies

combined_equity = INITIAL_CAPITAL * (1 + combined_returns).cumprod()

# Build a synthetic BacktestResult for the combined portfolio
combined_result = BacktestResult(
    strategy_name="Combined (equal-weight)",
    equity_curve=combined_equity,
    returns=combined_returns,
    positions=pd.DataFrame(),
    trades=pd.DataFrame(),
    metrics={},
    config={
        "initial_capital": INITIAL_CAPITAL,
        "benchmark": BENCHMARK_TICKER,
        "risk_free_rate": 0.0,
    },
)
combined_result.metrics = compute_metrics(combined_result, benchmark=spy_result)

# Diversification summary
best_sharpe = max(r.metrics.get("sharpe_ratio", 0.0) or 0.0 for r in all_results)
comb_sharpe = combined_result.metrics.get("sharpe_ratio", 0.0) or 0.0
comb_dd     = combined_result.metrics.get("max_drawdown_pct", 0.0) or 0.0
comb_alpha  = combined_result.metrics.get("alpha_pct", 0.0) or 0.0

print(f"Combined Sharpe:        {comb_sharpe:.3f}")
print(f"Combined max drawdown:  {comb_dd:.2f}%")
print(f"Combined alpha:         {comb_alpha:.2f}%")
print()
print(
    f"Diversification benefit: combined Sharpe {comb_sharpe:.3f} vs "
    f"best individual Sharpe {best_sharpe:.3f}"
)

## Figure 9 — Combined portfolio

In [ ]:
fig, ax = plot_equity_curves(
    all_results + [combined_result], spy_result,
    title="Individual strategies + combined portfolio vs SPY")
save_figure(fig, "09_combined_portfolio", FIGURES_DIR)
plt.show()

fig, ax = plot_monthly_returns_heatmap(combined_result)
save_figure(fig, "10_combined_monthly_returns", FIGURES_DIR)
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("=== BACKTEST SUMMARY ===")
print("=" * 60)
print(f"Period: {START_DATE} -> {END_DATE}")
print(f"Universe: {len(DEFAULT_TICKERS)} S&P 500 stocks + {BENCHMARK_TICKER} benchmark")
print(f"Initial capital: ${INITIAL_CAPITAL:,.0f}")
print()
print("Strategy performance (annualized):")

strategy_report = [
    ("Momentum",       mom_result),
    ("Mean Reversion", mr_result),
    ("ML Signal",      ml_result),
    ("Combined",       combined_result),
]

for label, r in strategy_report:
    ret    = r.metrics.get("annualized_return_pct") or float("nan")
    sharpe = r.metrics.get("sharpe_ratio")          or float("nan")
    dd     = r.metrics.get("max_drawdown_pct")      or float("nan")
    alpha  = r.metrics.get("alpha_pct")             or float("nan")
    print(
        f"  {label:<16} return={ret:.1f}%  Sharpe={sharpe:.2f}  "
        f"MaxDD={dd:.1f}%  Alpha={alpha:.1f}%"
    )

spy_ret    = spy_result.metrics.get("annualized_return_pct") or float("nan")
spy_sharpe = spy_result.metrics.get("sharpe_ratio")          or float("nan")
spy_dd     = spy_result.metrics.get("max_drawdown_pct")      or float("nan")
print(
    f"  {'SPY benchmark':<16} return={spy_ret:.1f}%  Sharpe={spy_sharpe:.2f}  "
    f"MaxDD={spy_dd:.1f}%"
)

print()
print("Correlation structure:")
strat_ret_dict = {r.strategy_name[:15]: r.returns for r in all_results}
corr_mat = pd.concat(strat_ret_dict, axis=1).dropna().corr()
print(corr_mat.round(3).to_string())

# Count exported files
n_files = sum(
    1 for p in FIGURES_DIR.glob("*")
    if p.suffix in (".png", ".pdf")
)
print(f"\nFigures exported to: {FIGURES_DIR} ({n_files} files)")

## Notes

- All figures are saved at 150 DPI in both PNG (for README preview) and PDF
  (for print/presentation quality).
- The ML signal strategy uses a 3-year minimum training window with quarterly
  retraining — this is the primary cause of its later start date.
- The combined portfolio uses a naive equal-weight average of daily returns;
  no rebalancing costs are modelled at the combination level.